In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import os
import gc
import math
import random
from typing import Union
from tqdm import tqdm
import wandb
from diffusers.training_utils import EMAModel
from diffusers.optimization import get_scheduler
from transformers import AutoTokenizer
from accelerate.utils import ProjectConfiguration, set_seed
from accelerate import Accelerator
from diffusers import (
    AutoencoderKL,
    DDPMScheduler,
    UNet2DConditionModel,
)
from torch.utils.data import DataLoader
import torch
import torch.nn.functional as F
from torch.utils.checkpoint import checkpoint

In [ ]:
from src.t2iadapter.config import (T2IConfig)
from src.t2iadapter.MRIProjector import MRIProjector, LatentMRIProjector
from src.t2iadapter.utils import (
    compute_embeddings_sd1x5,
    log_configs,
    print_trainable_parameters,
    generate_mri_slices_partial_latent_align_dc_no_t2i,
    import_model_class_from_model_name_or_path
)
from src.slicedMRI import DatasetConfig, FastMRILazyDataset
from src.slicedMRI.transform_to_2D_slices import build_fastMRI_manifest
from src.eval import MRIEvaluator
from torchmetrics.image import PeakSignalNoiseRatio, StructuralSimilarityIndexMeasure

In [ ]:
train_val_test_split = (0.8, 0.1, 0.1)
assert sum(train_val_test_split) == 1.0, "Dataset split should sum up to one"

shared_config: dict = {
    "data_dir": Path("./fastMRI_brain_DICOM"),
    "mode": "train",
    "fractions": train_val_test_split,
    "target_size": (512, 512),
    "contrast_filter": "T2",
    "strength_filter": "3.0T",
    "scale_factor": 4.0,
    "fastMRI_manifest_json": "./fast_MRI_brain_patient_records_manifest_small.json",
}
config = DatasetConfig(**shared_config)
t2i_config = T2IConfig(
    train_batch_size=32,
    test_batch_size=16,
    partial_start_step=999,
    max_train_steps=6500,
    pretrained_vae_model_name_or_path="microsoft/mri-autoencoder-v0.1",
)
# train_dataset = PairedMRI_MiniDataset(config=DatasetConfig(**shared_config), verbose=1)
train_dataset = FastMRILazyDataset(config=DatasetConfig(**shared_config))
shared_config["mode"] = "test"
test_dataset = FastMRILazyDataset(config=DatasetConfig(**shared_config))
shared_config["mode"] = "val"
val_dataset = FastMRILazyDataset(config=DatasetConfig(**shared_config))
# test_dataset = PairedMRI_MiniDataset(config=DatasetConfig(**shared_config), verbose=1)
train_loader = DataLoader(
    train_dataset,
    batch_size=t2i_config.train_batch_size,
    shuffle=True,
    num_workers=2,
    pin_memory=True,
)
test_loader = DataLoader(
    test_dataset,
    batch_size=t2i_config.test_batch_size,
    shuffle=True,
    num_workers=2,
    pin_memory=True,
)
val_loader = DataLoader(
    val_dataset,
    batch_size=t2i_config.test_batch_size,
    shuffle=True,
    num_workers=2,
    pin_memory=True,
)

### Training Setup

In [ ]:
accelerator_project_config = ProjectConfiguration(
    project_dir=t2i_config.output_dir, logging_dir=t2i_config.logging_dir
)
accelerator = Accelerator(
    gradient_accumulation_steps=t2i_config.gradient_accumulation_steps,
    mixed_precision=t2i_config.mixed_precision,  # set bf16
    log_with=t2i_config.report_to,
    project_config=accelerator_project_config,
)
set_seed(t2i_config.seed)
os.makedirs(t2i_config.output_dir, exist_ok=True)

weight_dtype = torch.float32
if accelerator.mixed_precision == "fp16":
    weight_dtype = torch.float16
elif accelerator.mixed_precision == "bf16":
    weight_dtype = torch.bfloat16
# load the tokenizers
tokenizer = AutoTokenizer.from_pretrained(
    t2i_config.pretrained_model_name_or_path,
    subfolder="tokenizer",
    revision=t2i_config.revision,
    use_fast=False,
)
# load the correct scheduler and models
text_encoder_cls = import_model_class_from_model_name_or_path(
    t2i_config.pretrained_model_name_or_path,
    t2i_config.revision,
    subfolder="text_encoder",
)
# Load scheduler and models
# timesteps defauls to 1000
noise_scheduler = DDPMScheduler.from_pretrained(
    t2i_config.pretrained_model_name_or_path,
    subfolder="scheduler",
    prediction_type=t2i_config.ddpm_scheduler_prediction_type,  # velocity prediction
    timestep_spacing=t2i_config.ddpm_scheduler_timestep_spacing,  # for zero-SNR
    rescale_betas_zero_snr=t2i_config.ddpm_scheduler_rescale_betas_zero_snr,  # enforces pure noise at t=1000
)
text_encoder = text_encoder_cls.from_pretrained(
    t2i_config.pretrained_model_name_or_path,
    subfolder="text_encoder",
    revision=t2i_config.revision,
    torch_dtype=weight_dtype,
)
vae_path = (
    t2i_config.pretrained_model_name_or_path
    if t2i_config.pretrained_vae_model_name_or_path is None
    else t2i_config.pretrained_vae_model_name_or_path
)
vae = AutoencoderKL.from_pretrained(
    vae_path,
    subfolder="vae" if t2i_config.pretrained_vae_model_name_or_path is None else None,
    revision=t2i_config.revision,
    torch_dtype=weight_dtype,
)

unet_config = UNet2DConditionModel.load_config(
    "segmind/tiny-sd",
    subfolder="unet"
)

new_config = dict(unet_config)
new_config['in_channels'] = 8 # Keep your 8-channel modification
unet = UNet2DConditionModel.from_config(new_config)

unet.train()
# These are never trained to circumvent mode collapse (see controlnet paper)
vae.requires_grad_(False)
text_encoder.requires_grad_(False)

print(
    f"Using VAE: {vae.config['_name_or_path']}, UNET: {unet}, Scheduler Steps: {noise_scheduler.config["num_train_timesteps"]}"
)

In [ ]:
if t2i_config.enable_xformers_memory_efficient_attention:
    import xformers # pyright: ignore[reportMissingImports]
    from packaging import version
    xformers_version = version.parse(xformers.__version__)
    if xformers_version == version.parse("0.0.16"):
        print(
            "xFormers 0.0.16 cannot be used for training in some GPUs. If you observe problems during training, please update xFormers to at least 0.0.17. See https://huggingface.co/docs/diffusers/main/en/optimization/xformers for more details."
        )
    unet.enable_xformers_memory_efficient_attention()
    print(f"Enabled efficient attention on UNET")
if t2i_config.gradient_checkpointing:
    unet.enable_gradient_checkpointing()
    print(f"Enabled gradient checkpointing on UNET")
# Enable TF32 for faster training on Ampere GPUs,
# cf https://pytorch.org/docs/stable/notes/cuda.html#tensorfloat-32-tf32-on-ampere-devices
if t2i_config.allow_tf32:
    torch.backends.cuda.matmul.allow_tf32 = True
    print(f"Enabled matmul.allow_tf32")
if t2i_config.scale_lr:
    learning_rate = (
        t2i_config.learning_rate
        * t2i_config.gradient_accumulation_steps
        * t2i_config.train_batch_size
        * accelerator.num_processes
    )
else:
    learning_rate = t2i_config.learning_rate
# Use 8-bit Adam for lower memory usage or to fine-tune the model in 16GB GPUs
if t2i_config.use_8bit_adam:
    try:
        import bitsandbytes as bnb # pyright: ignore[reportMissingImports]
    except ImportError:
        raise ImportError(
            "To use 8-bit Adam, please install the bitsandbytes library: `pip install bitsandbytes`."
        )
    optimizer_class = bnb.optim.AdamW8bit
    print(f"Enabled 8bit adam")
else:
    optimizer_class = torch.optim.AdamW

In [ ]:
is_special_vae = (
    t2i_config.pretrained_vae_model_name_or_path == "microsoft/mri-autoencoder-v0.1"
)
output_dims = 2 if is_special_vae else 3
mri_projector = MRIProjector(output_dims=output_dims).to(accelerator.device)
params_to_optimize = (
    list(unet.parameters())
    + list(mri_projector.parameters())
)
optimizer = optimizer_class(
    params_to_optimize,
    lr=learning_rate,
    betas=(t2i_config.adam_beta1, t2i_config.adam_beta2),
    weight_decay=t2i_config.adam_weight_decay,
    eps=t2i_config.adam_epsilon,
)

# For mixed precision training we cast the text_encoder and vae weights to half-precision
# as these models are only used for inference, keeping weights in full precision is not required.
weight_dtype = torch.float32
if accelerator.mixed_precision == "fp16":
    weight_dtype = torch.float16
elif accelerator.mixed_precision == "bf16":
    weight_dtype = torch.bfloat16

vae.to(accelerator.device, dtype=weight_dtype)
unet.to(accelerator.device, dtype=weight_dtype)
print("Models loaded. VAE in", weight_dtype)

Models loaded. VAE in torch.bfloat16


In [ ]:
# Let's first compute all the embeddings so that we can free up the text encoders from memory.
# text_encoders = [text_encoder]
tokenizers = [tokenizer]
gc.collect()
torch.cuda.empty_cache()
# Scheduler and math around the number of training steps.
# num_update_steps_per_epoch = math.ceil(len(train_dataloader) / args.gradient_accumulation_steps)
num_update_steps_per_epoch = math.ceil(1e7 / t2i_config.gradient_accumulation_steps)
if t2i_config.max_train_steps is None:
    t2i_config.max_train_steps = (
        t2i_config.num_train_epochs * num_update_steps_per_epoch
    )
lr_scheduler = get_scheduler(
    t2i_config.lr_scheduler_name,
    optimizer=optimizer,
    num_warmup_steps=t2i_config.lr_warmup_steps * accelerator.num_processes,
    num_training_steps=t2i_config.max_train_steps * accelerator.num_processes,
    num_cycles=t2i_config.lr_num_cycles,
    power=t2i_config.lr_power,
)
ema_unet = EMAModel(
    unet.parameters(),
    model_cls=unet.__class__,  # Custom class, passing it here helps with some utilities
    decay=0.9999,
)
# Prepare everything with our `accelerator`.
mri_projector, unet, optimizer, train_dataloader, lr_scheduler = accelerator.prepare(
    mri_projector, unet, optimizer, train_loader, lr_scheduler
)

print_trainable_parameters(unet, name="UNet")
print_trainable_parameters(mri_projector, name="MRI-Projector")

### Training

In [ ]:
run = wandb.init(
    entity="hannes-leonhard",
    project="mir-sr",
    config=log_configs(t2i_config=t2i_config, dataset_config=config),
)

In [ ]:
global_step = 0
first_epoch = 0
initial_global_step = 0
random.seed(t2i_config.seed)
vis_idx = random.randrange(0, len(val_dataset))
print(f"Using test entry {vis_idx} for training visualization")
psnr = PeakSignalNoiseRatio(data_range=1.0).to(accelerator.device)
ssim = StructuralSimilarityIndexMeasure(data_range=1.0).to(accelerator.device)

random.seed(t2i_config.seed)
progress_bar = tqdm(
    range(0, t2i_config.max_train_steps),
    initial=initial_global_step,
    desc="Steps",
    disable=True,
)
prompt_embeds: Union[torch.Tensor, None] = None
inference_scheduler = DDPMScheduler.from_config(noise_scheduler.config)

for epoch in range(first_epoch, t2i_config.num_train_epochs):
    if global_step > t2i_config.max_train_steps:
        print(f"Finished training, will stop now")
        break
    unet.train()
    mri_projector.train()
    for step, batch in enumerate(train_loader):
        if global_step > t2i_config.max_train_steps:
            print(f"Finished training, will stop now")
            break
        with accelerator.accumulate(unet):
            # Move text encoder to GPU only for inference, then offload
            text_encoder.to(accelerator.device)
            with torch.no_grad():
                prompt_embeds = compute_embeddings_sd1x5(
                    batch=batch,
                    proportion_empty_prompts=0.1,
                    text_encoders=[text_encoder],
                    tokenizers=[tokenizer],
                    accelerator=accelerator,
                )["prompt_embeds"]
            # Offload text encoder back to CPU immediately
            text_encoder.to("cpu")

            bsz = batch["hr"].shape[0]
            with torch.amp.autocast(device_type="cuda", dtype=weight_dtype):
                # Use projector for HR (Target) and LR (Condition)
                hr_rgb = mri_projector(batch["hr"].to(accelerator.device).float())
                condition = mri_projector(batch["lr"].to(accelerator.device).float())
                with torch.no_grad():  # ensure not graph is built for vae
                    latents = vae.encode(
                        hr_rgb.to(vae.dtype)
                    ).latent_dist.sample()
                    latents = latents * vae.config.scaling_factor
                    latents = latents.to(weight_dtype)
                    latents_lr = vae.encode(
                        condition.to(vae.dtype)
                    ).latent_dist.sample()
                    latents_lr = latents_lr * vae.config.scaling_factor
                    latents_lr = latents_lr.to(weight_dtype)
                # Noise generation
                noise = torch.randn_like(latents)
                # SDEEdit
                timesteps = torch.randint(
                    0,
                    t2i_config.partial_start_step,
                    (bsz,),
                    device=latents.device,
                ).long()
                noisy_latents = noise_scheduler.add_noise(
                    latents, noise, timesteps
                )
                unet_input = torch.cat([noisy_latents, latents_lr], dim=1)
                # Adapter conditioning
                model_pred = unet(
                    unet_input,
                    timesteps,
                    encoder_hidden_states=prompt_embeds,
                ).sample

                # Loss Computation
                if noise_scheduler.config.prediction_type == "epsilon":
                    loss = F.mse_loss(
                        model_pred.float(), noise.float(), reduction="mean"
                    )
                elif noise_scheduler.config.prediction_type == "v_prediction":
                    loss = F.mse_loss(
                        model_pred.float(),
                        noise_scheduler.get_velocity(
                            latents, noise, timesteps
                        ).float(),
                        reduction="mean",
                    )
                else:
                    raise ValueError(
                        f"Unknown prediction type {noise_scheduler.config.prediction_type}"
                    )
            accelerator.backward(loss)

            if accelerator.sync_gradients:
                params_to_clip = (
                    list(unet.parameters())
                    + list(mri_projector.parameters())
                )
                accelerator.clip_grad_norm_(params_to_clip, t2i_config.max_grad_norm)
                optimizer.step()
                lr_scheduler.step()
                if accelerator.is_main_process:
                    unwrapped_adapter = accelerator.unwrap_model(unet)
                    ema_unet.step(unwrapped_adapter.parameters())
                optimizer.zero_grad()
            # Delete heavy tensors
            del (
                model_pred,
                noisy_latents,
                condition,
                hr_rgb,
                latents,
            )
            if step % 50 == 0:
                gc.collect()
                torch.cuda.empty_cache()

        if accelerator.sync_gradients:
            progress_bar.update(1)
            global_step += 1

            # --- Validation / Visualization ---
            if accelerator.is_main_process:
                if global_step % t2i_config.media_reporting_step == 0:
                    item = val_dataset[vis_idx]
                    # Add batch dimension
                    item["hr"] = item["hr"].unsqueeze(0)
                    item["lr"] = item["lr"].unsqueeze(0)
                    item["txt"] = [item["txt"]]

                    with torch.no_grad():
                        text_encoder.to(accelerator.device)
                        prompt_embeds_eval = compute_embeddings_sd1x5(
                            batch=item,
                            proportion_empty_prompts=0.0,
                            text_encoders=[text_encoder],
                            tokenizers=tokenizers,
                            accelerator=accelerator,
                            is_train=False,
                        )["prompt_embeds"]
                        text_encoder.to("cpu")
                    images, _ = generate_mri_slices_partial_latent_align_dc_no_t2i(
                        batch=item,
                        mri_projector=mri_projector,
                        unet=unet,
                        vae=vae,
                        noise_scheduler=inference_scheduler,
                        prompt_embeds=prompt_embeds_eval,
                        start_step=t2i_config.partial_start_step,  # Start from t=250
                        num_inference_steps=500,  # Scheduler will slice this
                        weight_dtype=weight_dtype,
                        accelerator=accelerator,
                        use_data_consistency=True,
                        dc_reduction_factor=1.8,
                        taper=0.45,
                        apply_final_pixel_dc=True,
                    )
                    mri_projector.train()
                    unet.train()
                    views = []
                    images_gt = wandb.Image(
                        item["hr"].numpy()[0,:,:,:].transpose((1, 2, 0)),
                        caption="GT MRI",
                    )
                    images_lr = wandb.Image(
                        item["lr"].numpy()[0,:,:,:].transpose((1, 2, 0)),
                        caption="Low-Res MRI",
                    )
                    views.append(images_gt)
                    views.append(images_lr)
                    channels = images.shape[3]
                    for dim in range(channels):
                        images_gen = wandb.Image(
                            images[0, :, :, dim][:, :, np.newaxis],
                            caption=f"Gen MRI (axis {dim}) - Partial Diff",
                        )
                        views.append(images_gen)
                    run.log({"validation views": views})

                    # Metrics calculation
                    if item["hr"].ndim == 3:
                        gt = item["hr"].unsqueeze(1).expand(1, channels, 512, 512)
                    else:
                        gt = item["hr"].expand(1, channels, 512, 512)

                    metrics_val = MRIEvaluator.eval_all_metrics(
                        ground_truth=gt.to(accelerator.device),
                        generated=torch.from_numpy(images)
                        .permute(0, 3, 1, 2)
                        .to(accelerator.device),
                        psnr=psnr,
                        ssim=ssim,
                    )
                    run.log(
                        {
                            "val_hfen": metrics_val[0],
                            "val_nmse": metrics_val[1],
                            "val_psnr": metrics_val[2],
                            "val_ssim": metrics_val[3],
                        }
                    )

                if global_step % t2i_config.checkpointing_steps == 0:
                    save_path = os.path.join(
                        t2i_config.output_dir, f"checkpoint-{global_step}"
                    )
                    os.makedirs(save_path, exist_ok=True)
                    unwrapped_unet = accelerator.unwrap_model(unet)
                    ema_unet.store(unwrapped_unet.parameters())
                    ema_unet.copy_to(unwrapped_unet.parameters())
                    torch.save(
                        unwrapped_unet.state_dict(),
                        os.path.join(save_path, "adapter.pt"),
                    )
                    ema_unet.restore(unwrapped_unet.parameters())
                    torch.save(
                        mri_projector.state_dict(),
                        os.path.join(save_path, "mri_projector.pt"),
                    )
                    print(f"Saved EMA checkpoint to {save_path}")

        logs = {
            "loss": loss.detach().item(),
            "lr": lr_scheduler.get_last_lr()[0],
            "epoch": epoch,
        }
        run.log(logs)

final_out = "/content/drive/MyDrive/Colab Notebooks/MasterInfo/GenAI/scratch_noPartial_65k"
# After training is done
if accelerator.is_main_process:
    unwrapped_unet = accelerator.unwrap_model(unet)
    # Final EMA Save
    ema_unet.copy_to(unwrapped_unet.parameters())
    torch.save(
        unwrapped_unet.state_dict(),
        os.path.join(final_out, "adapter_tuned_scratch.pt"),
    )
    torch.save(
        mri_projector.state_dict(),
        os.path.join(final_out, "mri_projector_scratch.pt"),
    )
    print("Training finished. Saved final T2I-Adapter EMA.")

run.finish()